# Intervals, verdicts, typed failures, and test statistics

Three small types that enforce rule 4 ("every number carries its provenance") and rule 5
("never swallow an exception"), plus the statistics helpers the acceptance tests use.

In [ ]:
import numpy as np

from axiom.core import (
    AcceptanceRegion,
    DIGITS,
    Assumption,
    Blocked,
    Failure,
    Interval,
    LedgerLine,
    NonEmptyStr,
    Summary,
    Unsupported,
    Unverified,
    Verdict,
    clopper_pearson,
    decimals_for,
    effective_sample_size,
    eti,
    format_interval,
    format_measured,
    hdi,
    interval,
    is_failure,
    mc_standard_error,
    round_to,
    summarize,
    wald,
    z_score,
)

## Intervals carry their definition

An `Interval` cannot be constructed without `definition` and `mass`. The parent repo shipped
the same estimand at two masses and two definitions in two places before anyone noticed; here
that is a type error.

In [ ]:
from axiom.display import show

rng = np.random.default_rng(0)
draws = rng.gamma(2.0, 1.0, size=20_000)      # right-skewed, so HDI and ETI differ

h = hdi(draws, 0.9)
e = eti(draws, 0.9)
show(h)
show(e)
print("HDI is narrower on a skewed posterior:", h.width < e.width)

In [ ]:
try:
    Interval(lower=0.0, upper=1.0)  # no definition, no mass
except Exception as exc:
    print(type(exc).__name__, "- an interval without provenance cannot exist")

i = interval(draws, definition="eti", mass=0.5)
print(i, i.contains(2.0), i.width)

# a frequentist CI is a third kind, and says so
print(wald(estimate=1.2, se=0.3, mass=0.95))

In [ ]:
s: Summary = summarize(draws, definition="hdi", mass=0.95)
print(s)                                      # mean, sd and interval, all at one resolution
print(s.mean, s.median, s.sd, s.n)            # the stored values keep every digit
print(s.to_json()[:160], "...")

## A printed number stops where its uncertainty stops

A posterior mean of `2.0134729...` beside an interval half a unit wide is seven digits of
which two are knowledge. Printing the rest is not neutral — a reader takes trailing digits
as precision and compares two results on digits that are arithmetic. So every summary,
card, and report metric in axiom rounds the value to the place its own uncertainty reaches,
keeping `DIGITS` significant digits of that uncertainty.

This is a display rule and only a display rule: `io` still writes seventeen digits, and a
failure's `detail` still carries the number that reproduces the bug.

In [ ]:
print("digits kept of the uncertainty:", DIGITS)

for u in (0.52, 5.2, 52.0, 520.0):
    print(f"12.3456789 ± {u:<6} -> {format_measured(12.3456789, u):>8}   "
          f"decimals={decimals_for(u)}")

# with nothing to round against, nothing is rounded
print("no stated uncertainty ->", format_measured(12.3456789))

In [ ]:
# an interval is its own resolution: both bounds round to the half-width
print(format_interval(h.lower, h.upper), "from a half-width of", round(h.half_width, 4))
print(h)                                       # and the definition and mass come with it

# the numeric form, for a caller that keeps computing rather than printing
print(round_to(s.mean, s.sd), "vs the stored", s.mean)

## Verdicts and assumptions

`Verdict` is the shared status vocabulary — `identified | downgraded | blocked | unsupported
| unverified` — used by identification, transport, transfer plans, and design methods alike.
Its invariants are enforced at construction: a `blocked` verdict needs a reason, a
`downgraded` one needs at least one named `Assumption`, and an `identified` one cannot carry
an unverified assumption.

In [ ]:
overlap = Assumption(
    name="overlap",
    facet="population",
    statement="every target stratum has positive support in the source",
    challenged_by="propensity / dose overlap diagnostic",
)
print(overlap.state)

v = Verdict(status="downgraded", route="backdoor", assumptions=(overlap,))
print(v.status, v.licensed, [a.name for a in v.assumptions])

In [ ]:
for bad in (
    lambda: Verdict(status="blocked"),
    lambda: Verdict(status="downgraded"),
    lambda: Verdict(status="identified", assumptions=(overlap,)),
):
    try:
        bad()
    except ValueError as exc:
        print("refused:", exc)

print(Verdict(status="identified", assumptions=(overlap.satisfied(),)).status)
print(Verdict(status="blocked", reason="no admissible adjustment set").licensed)

A `LedgerLine` is one entry in the assumption ledger. It may carry an `Assumption` (something
that could be false) or just record a fact (a unit conversion). `source`/`target` hold the
content hashes of the specs on either side when there are two.

In [ ]:
line = LedgerLine(
    kind="facet:population",
    statement="transferred from region=north to region=all under overlap",
    assumption=overlap.asserted(),
    detail={"from": "north", "to": "all"},
)
print(line.kind, "|", line.assumption.state if line.assumption else None)

## Typed failures

`Unsupported`, `Blocked`, `Unverified` are *returned* by code that must not guess. They are
falsy, need a non-empty reason, and serialize. `is_failure` narrows a `float | Failure`.

In [ ]:
def realize(value_ok: bool) -> float | Failure:
    if not value_ok:
        return Blocked(reason="adjustment set contains an unmeasured variable", detail={"node": "U"})
    return 0.42


for ok in (True, False):
    r = realize(ok)
    if is_failure(r):
        print(r.status, "->", r.reason, r.detail)
    else:
        print("value", r)

print(bool(Unsupported(reason="no marginal capability", missing=("marginal",))), bool(Unverified(reason="overlap not checked")))

# the three share a field type, not a base class (composition over inheritance)
reason: NonEmptyStr = "a reason that is not blank"
try:
    Blocked(reason="   ")
except Exception as exc:
    print(type(exc).__name__, "- a blank reason is refused")

## Statistics for acceptance tests

Every rate criterion in the roadmap states N and an exact binomial acceptance region at
α = 0.001 (review B5). `clopper_pearson` produces it. `effective_sample_size` and
`mc_standard_error` give the Monte-Carlo error that statistical golden fixtures compare
against with `z_score` (review B6).

In [ ]:
region: AcceptanceRegion = clopper_pearson(n=500, p=0.05, alpha=0.001)
show(region)
print("count 25 accepted:", region.accepts(25), "| rate bounds:", region.rate_bounds)

In [ ]:
chains = rng.normal(size=(4, 1000))
ar = np.zeros_like(chains)
for c in range(4):
    for t in range(1, 1000):
        ar[c, t] = 0.8 * ar[c, t - 1] + 0.6 * rng.normal()

print("ESS iid   :", round(effective_sample_size(chains)))
print("ESS AR(1) :", round(effective_sample_size(ar)))
print("MCSE      :", mc_standard_error(ar))
print("z vs ref  :", z_score(value=ar.mean(), reference=0.0, reference_se=mc_standard_error(ar)))